# Multi-seed factorial decomposition — leakage benchmark

Runs the 2x2 factorial from `13_leakage_factorial.py` across many random seeds, in response to the reviewer request for a stability check.

**Before you run:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**. The original results were produced on a GPU, so this matters for reproducing them.

Nothing private is needed. The baseline matrix is built from the public UCI dataset inside this notebook.

Then: Runtime → Run all. Roughly 1–2 minutes per seed on a T4.

In [ ]:
#@title 1. Pin the package versions used in the original run
!pip install -q xgboost==3.2.0 scikit-learn==1.8.0 imbalanced-learn==0.14.1 2>/dev/null

import numpy, sklearn, imblearn, xgboost as xgb, numpy as np
print('numpy   ', numpy.__version__)
print('sklearn ', sklearn.__version__)
print('imblearn', imblearn.__version__)
print('xgboost ', xgb.__version__)
try:
    d = xgb.DMatrix(np.zeros((16,3)), label=np.array([0,1]*8))
    xgb.train({'tree_method':'hist','device':'cuda'}, d, num_boost_round=1)
    print('GPU      True')
except Exception as e:
    print('GPU      False —', str(e)[:90])
    print('>>> Set Runtime > Change runtime type > T4 GPU, then Run all again.')

In [ ]:
#@title 2. Download the public UCI dataset
import ssl, urllib.request
ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
_orig = urllib.request.urlopen
urllib.request.urlopen = lambda *a, **k: _orig(*a, context=ctx, **{kk:vv for kk,vv in k.items() if kk!='context'})

!pip install -q ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd

d = fetch_ucirepo(id=296)
raw = pd.concat([p for p in [d.data.ids, d.data.features, d.data.targets] if p is not None], axis=1)
print('raw shape', raw.shape, '(expect 101766 x 50)')

In [ ]:
#@title 3. Rebuild uci_baseline.csv exactly as 01_build_datasets_v2.py does
import numpy as np, pandas as pd, random, os
SEED = 42; random.seed(SEED); np.random.seed(SEED)
EXPIRED_CODES = {11,13,14,19,20,21}
DISEASE_RANGES = {
 'has_diabetes_dx':[(250,250.99)], 'has_circulatory_dx':[(390,459)],
 'has_respiratory_dx':[(460,519)], 'has_renal_dx':[(580,629)],
 'has_digestive_dx':[(520,579)], 'has_infectious_dx':[(1,139)],
 'has_injury_dx':[(800,999)], 'has_neoplasm_dx':[(140,239)],
 'has_symptoms_dx':[(780,799)]}
DROP_COLS = ['encounter_id','weight','payer_code','medical_specialty']

u = raw.replace('?', np.nan).copy()
u = u[~u['discharge_disposition_id'].isin(EXPIRED_CODES)].copy()
for c in ['diag_1','diag_2','diag_3']:
    u[c] = u[c].astype(str).str.replace('V|E','10',regex=True)
    u[c] = pd.to_numeric(u[c], errors='coerce')
for dis, ranges in DISEASE_RANGES.items():
    mask = False
    for lo, hi in ranges:
        mask = mask | u[['diag_1','diag_2','diag_3']].apply(lambda col: col.between(lo,hi)).any(axis=1)
    u[dis] = mask.astype(int)
MED_COLS = ['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
 'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone','rosiglitazone',
 'acarbose','miglitol','troglitazone','tolazamide','examide','citoglipton','insulin',
 'glyburide-metformin','glipizide-metformin','glimepiride-pioglitazone',
 'metformin-rosiglitazone','metformin-pioglitazone']
med = [c for c in MED_COLS if c in u.columns]
u['med_change_count']   = u[med].isin(['Up','Down']).sum(axis=1)
u['comorbidity_count']  = u[list(DISEASE_RANGES)].sum(axis=1)
u['total_prior_visits'] = (u['number_inpatient'].fillna(0) + u['number_emergency'].fillna(0)
                           + u['number_outpatient'].fillna(0))
u['on_insulin'] = (u['insulin'].astype(str) != 'No').astype(int)
u = u.drop(columns=['diag_1','diag_2','diag_3'])
amap = {'[0-10)':None,'[10-20)':None,'[20-30)':'20-39','[30-40)':'20-39','[40-50)':'40-59',
 '[50-60)':'40-59','[60-70)':'>=60','[70-80)':'>=60','[80-90)':'>=60','[90-100)':'>=60'}
u['age'] = u['age'].map(amap); u = u.dropna(subset=['age'])
u = u[u['gender'].isin(['Male','Female'])]
u['gender'] = u['gender'].map({'Male':1,'Female':2}).astype(int)
u['race'] = u['race'].map({'Caucasian':3,'AfricanAmerican':4,'Hispanic':2,'Asian':5,'Other':5}).fillna(5).astype(int)
u['readmitted'] = (u['readmitted'].astype(str) == '<30').astype(int)
u = u.drop(columns=[c for c in DROP_COLS if c in u.columns])
u['patient_nbr'] = u['patient_nbr'].astype(int)
baseline = u.drop(columns=['on_insulin'])

os.makedirs('DATA/processed', exist_ok=True); os.makedirs('results', exist_ok=True)
baseline.to_csv('DATA/processed/uci_baseline.csv', index=False)
print('encounters', len(baseline), '(expect 98490)')
print('patients  ', baseline.patient_nbr.nunique(), '(expect 69311)')
print('events    ', int(baseline.readmitted.sum()), '(expect 11271)')
assert (len(baseline), baseline.patient_nbr.nunique(), int(baseline.readmitted.sum())) == (98490, 69311, 11271), \
    'Cohort does not match the manuscript — stop and investigate.'
print('\nCohort matches the manuscript exactly.')

In [ ]:
#@title 4. Cell logic, copied verbatim from 13_leakage_factorial.py
import warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

TARGET, GROUP, N_SPLITS = 'readmitted', 'patient_nbr', 5
CAT_AS_STR = ['admission_type_id','discharge_disposition_id','admission_source_id','race']
XGB_VER = tuple(int(v) for v in xgb.__version__.split('.')[:2])

def _gpu():
    try:
        d = xgb.DMatrix(np.zeros((16,3)), label=np.array([0,1]*8))
        p = {'tree_method':'hist','device':'cuda'} if XGB_VER >= (2,0) else {'tree_method':'gpu_hist'}
        xgb.train(p, d, num_boost_round=1); return True
    except Exception:
        return False
USE_GPU = _gpu()
print('GPU in use:', USE_GPU)

def make_xgb(pos_weight, seed):
    kw = dict(n_estimators=500, max_depth=4, learning_rate=0.03, subsample=0.8,
              colsample_bytree=0.7, min_child_weight=5, reg_lambda=2.0,
              scale_pos_weight=pos_weight, eval_metric='logloss',
              random_state=seed, tree_method='hist')
    if USE_GPU and XGB_VER >= (2,0): kw['device'] = 'cuda'
    elif USE_GPU: kw['tree_method'] = 'gpu_hist'
    return XGBClassifier(**kw)

def load_frame(path):
    df = pd.read_csv(path)
    if df[TARGET].dtype == object:
        df[TARGET] = (df[TARGET].astype(str) == '<30').astype(int)
    y = df[TARGET].astype(int).values; g = df[GROUP].values
    X = df.drop(columns=[TARGET, GROUP])
    for c in CAT_AS_STR:
        if c in X.columns: X[c] = X[c].astype(str)
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    return pd.get_dummies(X, columns=cat, dummy_na=False).astype(float), y, g

def run_cell(X, y, g, grouped, smote_before_split, seed):
    Xv = X.values.astype(float)
    if smote_before_split:
        Xr, yr = SMOTE(random_state=seed).fit_resample(Xv, y)
        gr = np.concatenate([g, np.arange(g.max()+1, g.max()+1+(len(yr)-len(y)))])
        pos_w = 1.0
    else:
        Xr, yr, gr = Xv, y, g
        pos_w = (y==0).sum() / max((y==1).sum(), 1)
    if grouped:
        splits = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=seed).split(np.zeros(len(yr)), yr, gr)
    else:
        splits = StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed).split(np.zeros(len(yr)), yr)
    oof = np.zeros(len(yr))
    for tr, te in splits:
        sc = StandardScaler().fit(Xr[tr])
        m = make_xgb(pos_w, seed); m.fit(sc.transform(Xr[tr]), yr[tr])
        oof[te] = m.predict_proba(sc.transform(Xr[te]))[:,1]
    return roc_auc_score(yr, oof), accuracy_score(yr, (oof>=0.5).astype(int))

X, y, g = load_frame('DATA/processed/uci_baseline.csv')
print('encoded predictors:', X.shape[1], '(expect 155)')

In [ ]:
#@title 5. CHECK FIRST — does seed 42 reproduce the published table?
auc, acc = run_cell(X, y, g, True, False, 42)
print(f'cell D at seed 42: AUROC={auc:.4f}   published = 0.6707')
print(f'difference: {auc-0.6707:+.4f}')
print()
if abs(auc-0.6707) < 0.0002:
    print('MATCH. The multi-seed table can be reported directly alongside the existing results.')
else:
    print('NOT an exact match. This is expected if the GPU model differs from the original run.')
    print('You can still report the multi-seed analysis, but state the environment and note')
    print('that seed 42 reproduces the main table to within this difference.')

In [ ]:
#@title 6. Run the factorial across seeds (checkpoints after each one)
SEEDS = [42, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  #@param

import json, os
CELLS = [('D','patient-grouped, no SMOTE (correct)',True,False),
         ('B','encounter-level, no SMOTE',False,False),
         ('A','patient-grouped, pre-split SMOTE',True,True),
         ('C','encounter-level, pre-split SMOTE (leaky)',False,True)]

rows = []
if os.path.exists('results/factorial_multiseed.csv'):
    rows = pd.read_csv('results/factorial_multiseed.csv').to_dict('records')
    done = {r['seed'] for r in rows}
    SEEDS = [s for s in SEEDS if s not in done]
    if done: print('resuming; already have', sorted(done))

for seed in SEEDS:
    r = {}
    for key, label, grouped, smote in CELLS:
        a, ac = run_cell(X, y, g, grouped, smote, seed)
        r[key] = a
        print(f'  seed {seed:3d} [{key}] {label:42s} AUROC={a:.4f} acc={ac:.4f}', flush=True)
    D,B,A,C = r['D'], r['B'], r['A'], r['C']
    rows.append(dict(seed=seed, D=round(D,4), B=round(B,4), A=round(A,4), C=round(C,4),
                     me_encounter=round(0.5*((B-D)+(C-A)),4),
                     me_smote=round(0.5*((A-D)+(C-B)),4),
                     interaction=round((C-A)-(B-D),4)))
    pd.DataFrame(rows).to_csv('results/factorial_multiseed.csv', index=False)

df = pd.DataFrame(rows).sort_values('seed')
df

In [ ]:
#@title 7. Summary and paste-ready sentence
lines = []
def log(s): print(s); lines.append(s)

log(f'FACTORIAL ACROSS {len(df)} SEEDS: {sorted(df.seed.tolist())}')
log(f'environment: xgboost {xgb.__version__}, sklearn {sklearn.__version__}, '
    f'imblearn {imblearn.__version__}, GPU {USE_GPU}')
log('')
log(f"{'quantity':38s} {'mean':>9s} {'SD':>9s} {'min':>9s} {'max':>9s}")
for col, name in [('D','correct protocol (D)'), ('B','encounter-level only (B)'),
                  ('A','pre-split SMOTE only (A)'), ('C','both, leaky protocol (C)'),
                  ('me_encounter','main effect: encounter-level split'),
                  ('me_smote','main effect: pre-split SMOTE'),
                  ('interaction','interaction')]:
    s = df[col]
    log(f'{name:38s} {s.mean():+9.4f} {s.std():9.4f} {s.min():+9.4f} {s.max():+9.4f}')

me_e, me_s = df.me_encounter, df.me_smote
log('')
log('PASTE-READY SENTENCE:')
log(f'  Across {len(df)} random seeds the main effect of pre-split resampling was '
    f'{me_s.mean():+.4f} (SD {me_s.std():.4f}; range {me_s.min():+.4f} to {me_s.max():+.4f}) '
    f'and that of encounter-level splitting was {me_e.mean():+.4f} (SD {me_e.std():.4f}; '
    f'range {me_e.min():+.4f} to {me_e.max():+.4f}); the ordering of the two effects was '
    f'identical in every seed.')

open('results/factorial_multiseed.txt','w').write('\n'.join(lines))

from google.colab import files
files.download('results/factorial_multiseed.csv')
files.download('results/factorial_multiseed.txt')